# Cross-Lingual Automated Essay Scoring  
## German → Spanish Zero-Shot Transfer using XLM-R

In this notebook, we evaluate the cross-lingual transfer capability of a German-trained
Automated Essay Scoring (AES) model on Spanish learner essays.
The model is evaluated in a zero-shot setting, meaning no additional fine-tuning
is performed on Spanish data.


In [1]:
import numpy as np
import pandas as pd
import torch
import re


from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from datasets import Value

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr


/Users/hassan/anaconda3/envs/aes-mps/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:


if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device)


Using device: mps


## Loading the German-Trained AES Model

The model was trained on German learner essays using XLM-R in a regression setting.
We reuse the same checkpoint for zero-shot evaluation on Spanish essays.


In [4]:
MODEL_DIR = "/Users/hassan/Desktop/NLP/Project/aes_models_and_data/xlmr_german_aes_best"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR,
    num_labels=1,
    problem_type="regression"
)


model.to(device)

print("German AES model loaded successfully.")


The tokenizer you are loading from '/Users/hassan/Desktop/NLP/Project/aes_models_and_data/xlmr_german_aes_best' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


German AES model loaded successfully.


## Loading the Spanish AES Dataset

We evaluate the German-trained model on Spanish learner essays.
The dataset contains cleaned essay text and discrete proficiency-based scores.


In [16]:


file_path = "/Users/hassan/Desktop/NLP/Project/aes_models_and_data/aes_spanish.tsv"

df = pd.read_csv(
    file_path,
    sep="\t",
    engine="python",
    comment="#",
    encoding="utf-8"
)

print(df.shape)
print(df.columns.tolist())


(3034, 41)
['Subcorpus', 'Filename', 'Year data collection', 'Placement test score (raw)', 'Placement test score (%)', 'Proficiency', 'Sex', 'Age', 'School/University/Institution', 'Major', 'Year at university/school', 'L1', "Father's native language", "Mother's nativelanguage", 'Languages spoken at home', 'Age of exposure to Spanish', 'Years studying Spanish', 'Stay abroad in Spanish speaking country (>= 1 month)', 'Stay abroad (where)', 'Stay abroad (when)', 'Stay abroad (months)', 'Language certificates (type and level)', 'Proficiency (self-assessment) speaking', 'Proficiency (self-assessment) listening', 'Proficiency (self-assessment) reading', 'Proficiency (self-assessment) writing', 'Proficiency (self-assessment)', 'Additional foreign language(s)', 'Proficiency (self-assessment) in additional language speaking', 'Proficiency (self-assessment) in additional language listening', 'Proficiency (self-assessment) in additional language reading', 'Proficiency (self-assessment) in additi

In [18]:


df = df[df["Text"].notna()].copy()

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"<br\s*/?>", " ", text)  
    text = re.sub(r"\s+", " ", text)        
    return text.strip()

df["text_clean"] = df["Text"].apply(clean_text)

df = df[df["text_clean"].str.len() > 30].copy()

df["score_raw"] = (
    df["Proficiency (self-assessment)"]
    .astype(str)
    .str.replace("/ 6", "", regex=False)
    .str.strip()
)

df["score_raw"] = pd.to_numeric(df["score_raw"], errors="coerce")


df = df[df["score_raw"].notna()].copy()

print("After cleaning:")
print(df.shape)
df[["text_clean", "score_raw"]].head(2)

After cleaning:
(3034, 43)


,text_clean,score_raw
0,"Aquel día, había un hombre que siempre pasaba ...",3.0
1,Había un hombre que estaba andando en la calle...,3.5


In [19]:
df_es = df[["text_clean", "score_raw"]].rename(
    columns={
        "text_clean": "text",
        "score_raw": "score"
    }
)

df_es = df_es.dropna()
df_es.reset_index(drop=True, inplace=True)

print(df_es.shape)
df_es.head()


(3034, 2)


,text,score
0,"Aquel día, había un hombre que siempre pasaba ...",3.00
1,Había un hombre que estaba andando en la calle...,3.50
2,Uno dejo' su bebé en la calle. Viene un camina...,4.75
3,El video cuenta las acciones de quitar un bebé...,2.25
4,Charlie estaba caminando en la calle cuando de...,3.50


## Preparing the Spanish Test Set

For fair comparison, we evaluate only on the held-out Spanish test split.
We reproduce the same split strategy used earlier.


In [24]:

# Mapping Spanish scores to AES scale

MIN_SCORE = 1
MAX_SCORE = 6

df["score"] = (
    df["score_raw"]
    .round()                    # round 3.5 → 4
    .clip(MIN_SCORE, MAX_SCORE)
    .astype(int)
)

print(df["score"].value_counts().sort_index())


score
1     266
2     651
3     433
4    1174
5     351
6     159
Name: count, dtype: int64


In [25]:
test_df = df.sample(frac=0.2, random_state=42).reset_index(drop=True)

print("Spanish test size:", test_df.shape)


Spanish test size: (607, 44)


## Tokenizing Spanish Essays

The same tokenizer used during German training is applied to Spanish text.
This ensures a fair cross-lingual comparison.


In [32]:


test_ds = Dataset.from_pandas(
    test_df[["text_clean", "score"]]
)

def tokenize(batch):
    return tokenizer(
        batch["text_clean"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )

test_ds = test_ds.map(tokenize, batched=True)
test_ds = test_ds.rename_column("score", "labels")
# Ensure labels are float for regression (required for MSE loss on MPS)
test_ds = test_ds.cast_column("labels", Value("float32"))
test_ds.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)




Casting the dataset: 100%|██████████| 607/607 [00:00<00:00, 11989.65 examples/s]


## Evaluation Metrics

We report:
- Quadratic Weighted Kappa (QWK)
- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)
- Pearson Correlation

These metrics are standard in AES literature.


In [35]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = preds.squeeze(-1)

    preds_round = np.clip(
        np.round(preds),
        MIN_SCORE,
        MAX_SCORE
    )

    return {
        "qwk": cohen_kappa_score(labels, preds_round, weights="quadratic"),
        "mae": mean_absolute_error(labels, preds_round),
        "rmse": mean_squared_error(labels, preds) ** 0.5,
        "pearson": pearsonr(labels, preds)[0],
    }


## Zero-Shot Evaluation: German → Spanish

The German-trained model is evaluated directly on Spanish essays
without any additional fine-tuning.


In [36]:
args = TrainingArguments(
    output_dir="./tmp_eval",
    per_device_eval_batch_size=16,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics
)

results = trainer.evaluate()
results


/Users/hassan/anaconda3/envs/aes-mps/lib/python3.10/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 1.308645486831665,
 'eval_model_preparation_time': 0.0063,
 'eval_qwk': 0.3509875688163835,
 'eval_mae': 0.8237232565879822,
 'eval_rmse': 1.143960439364782,
 'eval_pearson': 0.5072523355484009,
 'eval_runtime': 81.0844,
 'eval_samples_per_second': 7.486,
 'eval_steps_per_second': 0.469}